In [1]:
dataset = [
    {"Age": "young", "Spectacle_prescription": "myope", "Astigmatism": "no", "Tear_production_rate": "reduced", "Recommended_lenses": "none"},
    {"Age": "young", "Spectacle_prescription": "myope", "Astigmatism": "no", "Tear_production_rate": "normal", "Recommended_lenses": "soft"},
    {"Age": "young", "Spectacle_prescription": "myope", "Astigmatism": "yes", "Tear_production_rate": "reduced", "Recommended_lenses": "none"},
    {"Age": "young", "Spectacle_prescription": "myope", "Astigmatism": "yes", "Tear_production_rate": "normal", "Recommended_lenses": "hard"},
    {"Age": "young", "Spectacle_prescription": "hypermetrope", "Astigmatism": "no", "Tear_production_rate": "reduced", "Recommended_lenses": "none"},
    {"Age": "young", "Spectacle_prescription": "hypermetrope", "Astigmatism": "no", "Tear_production_rate": "normal", "Recommended_lenses": "soft"},
    {"Age": "young", "Spectacle_prescription": "hypermetrope", "Astigmatism": "yes", "Tear_production_rate": "reduced", "Recommended_lenses": "none"},
    {"Age": "young", "Spectacle_prescription": "hypermetrope", "Astigmatism": "yes", "Tear_production_rate": "normal", "Recommended_lenses": "hard"},

    {"Age": "pre-presbyopic", "Spectacle_prescription": "myope", "Astigmatism": "no", "Tear_production_rate": "reduced", "Recommended_lenses": "none"},
    {"Age": "pre-presbyopic", "Spectacle_prescription": "myope", "Astigmatism": "no", "Tear_production_rate": "normal", "Recommended_lenses": "soft"},
    {"Age": "pre-presbyopic", "Spectacle_prescription": "myope", "Astigmatism": "yes", "Tear_production_rate": "reduced", "Recommended_lenses": "none"},
    {"Age": "pre-presbyopic", "Spectacle_prescription": "myope", "Astigmatism": "yes", "Tear_production_rate": "normal", "Recommended_lenses": "hard"},
    {"Age": "pre-presbyopic", "Spectacle_prescription": "hypermetrope", "Astigmatism": "no", "Tear_production_rate": "reduced", "Recommended_lenses": "none"},
    {"Age": "pre-presbyopic", "Spectacle_prescription": "hypermetrope", "Astigmatism": "no", "Tear_production_rate": "normal", "Recommended_lenses": "soft"},
    {"Age": "pre-presbyopic", "Spectacle_prescription": "hypermetrope", "Astigmatism": "yes", "Tear_production_rate": "reduced", "Recommended_lenses": "none"},
    {"Age": "pre-presbyopic", "Spectacle_prescription": "hypermetrope", "Astigmatism": "yes", "Tear_production_rate": "normal", "Recommended_lenses": "none"},

    {"Age": "presbyopic", "Spectacle_prescription": "myope", "Astigmatism": "no", "Tear_production_rate": "reduced", "Recommended_lenses": "none"},
    {"Age": "presbyopic", "Spectacle_prescription": "myope", "Astigmatism": "no", "Tear_production_rate": "normal", "Recommended_lenses": "none"},
    {"Age": "presbyopic", "Spectacle_prescription": "myope", "Astigmatism": "yes", "Tear_production_rate": "reduced", "Recommended_lenses": "none"},
    {"Age": "presbyopic", "Spectacle_prescription": "myope", "Astigmatism": "yes", "Tear_production_rate": "normal", "Recommended_lenses": "hard"},
    {"Age": "presbyopic", "Spectacle_prescription": "hypermetrope", "Astigmatism": "no", "Tear_production_rate": "reduced", "Recommended_lenses": "none"},
    {"Age": "presbyopic", "Spectacle_prescription": "hypermetrope", "Astigmatism": "no", "Tear_production_rate": "normal", "Recommended_lenses": "soft"},
    {"Age": "presbyopic", "Spectacle_prescription": "hypermetrope", "Astigmatism": "yes", "Tear_production_rate": "reduced", "Recommended_lenses": "none"},
    {"Age": "presbyopic", "Spectacle_prescription": "hypermetrope", "Astigmatism": "yes", "Tear_production_rate": "normal", "Recommended_lenses": "none"}
]

In [2]:
# Configurações iniciais da Classe Alvo e Atributos Disponíveis
target_class = "soft"
current_data = dataset.copy()
available_attrs = ['Age', 'Spectacle_prescription', 'Astigmatism', 'Tear_production_rate']
rule_antecedents = []

step = 1

# Loop principal para encontrar a primeira regra do algoritmo PRISM
while True:
    # Formata a string do antecedente para ficar igual ao seu PDF
    ante_str = ' vazio' if not rule_antecedents else ' ' + ' AND '.join([a + ' = ' + v for a, v in rule_antecedents])
    print(f"No subconjunto atual ({len(current_data)} exemplos), partimos do antecedente{ante_str.replace('_', ' ')} e calculamos p({target_class} | atributo = valor) para todos os pares ainda disponíveis.")
    print("atributo = valor\t\tp(soft | atributo = valor)\tcobertura positiva")

    candidates = []

    # 1. Calcular as probabilidades
    for attr in available_attrs:
        # Encontra valores únicos que o atributo possui no dataset atual
        values = set([row[attr] for row in current_data])

        for val in values:
            subset = [row for row in current_data if row[attr] == val]
            if len(subset) == 0: continue

            # Verifica a cobertura: quantos exemplos desse subconjunto pertencem à classe soft?
            pos_cov = len([row for row in subset if row['Recommended_lenses'] == target_class])
            total = len(subset)
            p = pos_cov / total
            candidates.append({'attr': attr, 'val': val, 'p': p, 'pos': pos_cov, 'total': total})

    # 2. Ordena os candidatos: Maior probabilidade (p) primeiro, e se empatar, maior cobertura (pos)
    candidates.sort(key=lambda x: (x['p'], x['pos']), reverse=True)

    # Exibe as estatísticas
    for c in candidates:
        print(f"{c['attr'].replace('_', ' ')} = {c['val']}\t\t{c['pos']}/{c['total']} = {c['p']:.2f}\t\t\t{c['pos']}")

    # 3. Pega o vencedor
    best = candidates[0]
    best_attr_disp = best['attr'].replace('_', ' ')
    print(f"Escolha do passo {step}: {best_attr_disp} = {best['val']} ({best['pos']}/{best['total']} = {best['p']:.2f}).\n")

    # 4. Atualiza os dados e a regra
    rule_antecedents.append((best['attr'].replace('_', ' '), best['val']))
    # Filtra o dataset para a próxima iteração
    current_data = [row for row in current_data if row[best['attr']] == best['val']]
    # Remove o atributo já usado
    available_attrs.remove(best['attr'])

    # 5. Condição de parada da regra: precisão é 100% (1.0) ou atributos acabaram
    if best['p'] == 1.0 or not available_attrs:
        break

    step += 1

# Imprime o resultado final
final_rule_str = ' AND '.join([a + ' = ' + v for a, v in rule_antecedents])
print(f"IF {final_rule_str} THEN lenses = {target_class}")
print(f"Regra final com cobertura de {len(current_data)} exemplos: {target_class}={len(current_data)}. Como a regra está perfeita no subconjunto final, os exemplos cobertos são removidos antes de procurar a próxima regra.")

No subconjunto atual (24 exemplos), partimos do antecedente vazio e calculamos p(soft | atributo = valor) para todos os pares ainda disponíveis.
atributo = valor		p(soft | atributo = valor)	cobertura positiva
Astigmatism = no		5/12 = 0.42			5
Tear production rate = normal		5/12 = 0.42			5
Spectacle prescription = hypermetrope		3/12 = 0.25			3
Age = pre-presbyopic		2/8 = 0.25			2
Age = young		2/8 = 0.25			2
Spectacle prescription = myope		2/12 = 0.17			2
Age = presbyopic		1/8 = 0.12			1
Astigmatism = yes		0/12 = 0.00			0
Tear production rate = reduced		0/12 = 0.00			0
Escolha do passo 1: Astigmatism = no (5/12 = 0.42).

No subconjunto atual (12 exemplos), partimos do antecedente Astigmatism = no e calculamos p(soft | atributo = valor) para todos os pares ainda disponíveis.
atributo = valor		p(soft | atributo = valor)	cobertura positiva
Tear production rate = normal		5/6 = 0.83			5
Spectacle prescription = hypermetrope		3/6 = 0.50			3
Age = pre-presbyopic		2/4 = 0.50			2
Age = young		2/4